In [1]:

import geopandas as gpd
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import contextily

from sklearn.metrics import classification_report

import sys
sys.path.append('..')
from model.dataset import LABEL_MAPPING

CLASS_INDICES = [value for key, value in LABEL_MAPPING.items()]
CLASS_NAMES = [key for key in LABEL_MAPPING.keys()]


In [2]:

df = pd.read_csv('../data/labels/dataset_v3/labels.csv')

df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])

gdf = gpd.GeoDataFrame(df)

gdf_train = gdf.loc[gdf.split == 'train']

gdf_val = gdf.loc[gdf.split == 'val']

In [3]:
def get_kde(df, **kwargs):

    # Step 2: Extract x and y coordinates
    x = df.geometry.x.values
    y = df.geometry.y.values
    data = np.vstack([x, y])

    # Step 3: Compute KDE
    kde = gaussian_kde(data, **kwargs)
    
    return kde

In [4]:
classes = gdf.driver.unique()

In [5]:
kde_per_class = {}

for _cls in classes:
    gdf_class = gdf_train.loc[gdf_train.driver == _cls]
    
    class_kde = get_kde(gdf_class)
    kde_per_class[_cls] = class_kde

In [7]:
kde_per_class['mining']

In [76]:
preds = []

for idx, row in gdf_val.iterrows():
    kde_in_loc = []
    for _cls in classes:
        point_coords = np.vstack([row.geometry.x, row.geometry.y])
        
        kde_in_loc.append(kde_per_class[_cls](point_coords))
    
    
    idx = np.array(kde_in_loc).squeeze().argmax()
    preds.append(classes[idx])
    #densities = torch.Tensor(kde_in_loc).squeeze()
    #preds = F.softmax(densities)
    #for i in range(len(classes)):
    #    print(classes[i], preds[i])
    
        
    


In [77]:
targets = gdf_val.driver.to_list()


In [78]:
preds_ind = [LABEL_MAPPING[i] for i in preds]
targets_ind = [LABEL_MAPPING[i] for i in targets]


In [79]:
metrics = classification_report(targets_ind, preds_ind, output_dict=True,labels=CLASS_INDICES,target_names=CLASS_NAMES,zero_division=0)




In [82]:
pd.DataFrame(metrics).T['f1-score'].to_dict()

{'pasture': 0.4237165582067969,
 'forest plantation': 0.045207956600361664,
 'soy': 0.8578728840625999,
 'coffee': 0.07246376811594205,
 'palm oil': 0.6448893572181243,
 'mining': 0.18608621425522834,
 'urban': 0.5453047775947282,
 'cocoa': 0.9988700564971752,
 'rubber': 0.7730147575544625,
 'forest regrowth': 0.29644533869885986,
 'shifting agriculture': 0.8275862068965517,
 'accuracy': 0.5658204438466712,
 'macro avg': 0.5155870796091664,
 'weighted avg': 0.5172570906810985}

### Produce a kde.argmax() GeoTiff over the entire study area

In [92]:
xmin, ymin, xmax, ymax = gdf.total_bounds

In [35]:
kde_in_loc = []

# Step 4: Create a grid over which to evaluate the KDE
xmin, ymin, xmax, ymax = gdf.total_bounds
xgrid = np.linspace(xmin, xmax, 1000)  #More longitude than latitude, approx 5x
ygrid = np.linspace(ymin, ymax, 200)
X, Y = np.meshgrid(xgrid, ygrid)
grid_coords = np.vstack([X.ravel(), Y.ravel()])

for _cls in classes:
    print(_cls)
    # Step 5: Evaluate the KDE on the grid
    #Z = kde(grid_coords).reshape(X.shape)
    kde_in_loc.append(kde_per_class[_cls](grid_coords))


#idx = np.array(kde_in_loc).squeeze().argmax()


cocoa
coffee
forest plantation
forest regrowth
mining
palm oil
pasture
rubber
shifting agriculture
soy
urban


In [36]:
densities = np.array(kde_in_loc).squeeze()

In [44]:
densities.argmax(axis=0).reshape(X.shape).shape

(200, 1000)

In [37]:
most_likely_cls = densities.argmax(axis=0).reshape(X.shape)

In [38]:
import rasterio
from rasterio.transform import from_bounds
import numpy as np

# Define the bounding box (left, bottom, right, top) and the array
array = most_likely_cls.astype('int8')  
output_filepath = '../data/tmp/kde.tif'

# Number of rows and columns in your array
rows, cols = array.shape

# Define the transform based on the bounding box and array dimensions
transform = from_bounds(xmin, ymin, xmax, ymax, cols, rows)

# Define the metadata
metadata = {
    'driver': 'GTiff',  # GeoTIFF
    'height': rows,
    'width': cols,
    'count': 1,  # Number of bands
    'dtype': str(array.dtype),
    'crs': '+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs',  # WGS84 Coordinate Reference System
    'transform': transform
}

# Write the dataset
with rasterio.open(output_filepath, 'w', **metadata) as dst:
    dst.write(array, 1)  # Writing the array data to the first band